In [2]:
from collections import defaultdict
import json
import os
import re


In [3]:
data_dir = "../data"
raw_dir = os.path.join(data_dir, "raw")


In [4]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)


def load_tests(base_dir: str):
	data = {
		"pre": {"code": None, "full": None},
		"post": {"code": None, "full": None},
	}

	for phase in ["Pre", "Post"]:
		phase_key = phase.lower()
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				file_path = os.path.join(phase_dir, file)
				content = load_json(file_path)            

				kind = "code" if "code" in file else "full"
				data[phase_key][kind] = content

	return data


In [5]:
def parse_likert(v: str):
	if isinstance(v, str):
		m = re.match(r"AO0?(\d)", v)
		if m:
			return int(m.group(1))
	return None

def clean_code_data(data: dict):
	valid_users = set()

	for user_id, user_data in data.items():
		if user_data:
			items = {
				k: parse_likert(v)
				for k, v in user_data.items()
				if k.startswith("G02Q03[SQ")
			}

			if any(v is not None for v in items.values()):
				valid_users.add(user_id)
	
	deleted_users = set(data.keys()) - valid_users
	return valid_users, deleted_users


In [6]:
def save_json(path: str, data):
    if not path.endswith(".json"):
        path += ".json"
    
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


def filter_users(data: dict, valid_users: set):
    return {user_id: user_data for user_id, user_data in data.items() if user_id in valid_users}


In [7]:
processed_dir = os.path.join(data_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)


In [12]:
all_valid_users = None

for session_name in os.listdir(raw_dir):
	session_dir = os.path.join(raw_dir, session_name)

	tests = load_tests(session_dir)

	pre_code = tests["pre"]["code"]
	post_code = tests["post"]["code"]

	valid_pre, deleted_pre = clean_code_data(pre_code)
	valid_post, deleted_post = clean_code_data(post_code)

	session_valid = valid_pre & valid_post

	if all_valid_users is None:
		all_valid_users = session_valid
	else:
		all_valid_users |= session_valid

	valid_users = valid_pre & valid_post
	deleted_users = deleted_pre & deleted_post

	print(f"\nSession: {session_name}")
	print(f"Valid users: {len(valid_users)}")
	print(f"Deleted users: {len(deleted_users)}")

	processed_session_dir = os.path.join(processed_dir, session_name)
	os.makedirs(processed_session_dir, exist_ok=True)

	for phase_name, phase_data in tests.items():
		phase_dir = os.path.join(processed_session_dir, phase_name.capitalize())
		os.makedirs(phase_dir, exist_ok=True)

		for kind_name, kind_data in phase_data.items():
			path = os.path.join(phase_dir, f"{kind_name}.json")
			cleaned  = filter_users(kind_data, valid_users)
			save_json(path, cleaned)

print("\nGlobal valid users:", len(all_valid_users))



Session: CarpeDiem-11-06-2025
Valid users: 33
Deleted users: 21

Session: CarpeDiem-20-05-2025
Valid users: 48
Deleted users: 28

Global valid users: 81


In [9]:
merged_data = defaultdict(dict)

for session_name in os.listdir(raw_dir):
    session_dir = os.path.join(raw_dir, session_name)
    tests = load_tests(session_dir)

    for phase_name, phase_data in tests.items():
        for kind_name, kind_data in phase_data.items():
            key = (phase_name, kind_name)

            merged_data[key].update(kind_data)
            
print(len(merged_data[("pre", "code")]))
            

140


In [10]:
merged_dir = os.path.join(processed_dir, "merged")
os.makedirs(merged_dir, exist_ok=True)

for (phase_name, kind_name), data in merged_data.items():
    phase_dir = os.path.join(merged_dir, phase_name.capitalize())
    os.makedirs(phase_dir, exist_ok=True)
    
    cleaned = filter_users(data, all_valid_users)
    
    path = os.path.join(phase_dir, f"{kind_name}.json")
    save_json(path, cleaned)
    